In [1]:
import torch
from mmasim_kernels.nv_ptx.ada_lovelace import mma_kernels

torch.manual_seed(0)
HMMA = mma_kernels["m16n8k16.f32.f16.f16.f32"]
QMMA = mma_kernels["m16n8k16.f32.e5m2.e5m2.f32"]

In [2]:
bsz = 100
A = torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16).to(torch.float8_e5m2)
B = torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16).to(torch.float8_e5m2)
A, B

(tensor([[[-0.8750, -0.4375, -2.5000,  ..., -0.2188, -0.3125, -0.1875],
          [-1.2500, -0.6250, -0.6250,  ...,  1.2500,  3.5000, -0.8750],
          [ 0.2500,  1.0000, -0.2500,  ...,  0.2500,  0.1562, -0.2188],
          ...,
          [ 0.4375, -0.6250, -0.6250,  ..., -0.5000, -0.7500,  0.8750],
          [-0.6250,  0.8750,  0.0391,  ...,  0.3750,  0.7500, -0.0781],
          [ 0.0547, -0.1875, -0.3125,  ..., -0.1250, -0.4375, -3.0000]],
 
         [[ 0.1250,  0.2500,  0.8750,  ...,  0.6250, -1.2500, -0.5000],
          [-0.2188, -0.2188,  1.7500,  ..., -0.0234,  0.7500,  1.5000],
          [-1.5000, -0.3125,  0.7500,  ..., -0.3750,  1.2500,  0.2188],
          ...,
          [-0.2188, -0.4375, -0.6250,  ..., -0.4375,  0.0938, -0.4375],
          [ 1.0000, -0.1875, -0.0938,  ...,  0.6250,  0.6250, -0.6250],
          [ 1.5000, -0.5000,  0.5000,  ..., -0.5000, -2.0000,  0.7500]],
 
         [[-0.8750, -1.2500,  1.2500,  ..., -1.2500, -0.3750,  0.8750],
          [-0.7500,  0.2188,

In [3]:
D_HMMA = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_QMMA = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_real = A.double() @ B.double()
for t in range(bsz):
    for i in range(0, 128, 16):
        for j in range(0, 128, 8):
            for k in range(0, 128, 16):
                D_QMMA[t, i:i+16, j:j+8] = QMMA(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_QMMA[t, i:i+16, j:j+8])
                D_HMMA[t, i:i+16, j:j+8] = HMMA(A[t, i:i+16, k:k+16].half(), B[t, k:k+16, j:j+8].half(), D_HMMA[t, i:i+16, j:j+8])

In [4]:
print("QMMA MSE:", (D_real - D_QMMA).square().mean().item())
print("HMMA MSE:", (D_real - D_HMMA).square().mean().item())

QMMA MSE: 1.0269521427642737e-06
HMMA MSE: 4.221506557372433e-15
